<a href="https://colab.research.google.com/github/mikailachmad/nlp-trio-bertiga/blob/main/src/cleaning/Cleaning_article.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cleaning & Preprocessing Artikel CNBC (v2)

Input: `cnbc_articles_final.csv` (hasil scraping). Output: `cnbc_articles_clean.csv`.

Tujuan akhirnya adalah fitur NLP untuk memprediksi USD/IDR, dan di Tugas 2 kita tidak boleh memakai transformer atau embedding pre-trained. Jadi teks disiapkan dalam dua versi: satu untuk lexicon sentimen (VADER), satu untuk TF-IDF/BoW dan kamus Loughran-McDonald.

Urutan: baca, buang failed, filter periode, cleaning, filter noise, filter relevansi geopolitik, outlier panjang, duplikat, buat dua versi teks, simpan. Jumlah artikel di tiap tahap dicatat untuk laporan.

In [3]:
from google.colab import files

INPUT_FILE = "cnbc_articles_final.csv"

print(f"Please upload the file: {INPUT_FILE}")
uploaded = files.upload()

Please upload the file: cnbc_articles_final.csv


Saving cnbc_articles_final.csv to cnbc_articles_final.csv


In [4]:
import re
import html
import unicodedata
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS

INPUT_FILE = "cnbc_articles_final.csv"
OUTPUT_FILE = "cnbc_articles_clean.csv"

# periode waktu (dihitung dalam WIB)
START_DATE = "2021-09-01"
END_DATE = "2026-09-01"

MIN_PARAGRAPH_WORDS = 4      # minimal jumlah kata dalam 1 paragraf
GEO_MIN_HITS = 3             # minimal istilah geopolitik berbeda di judul+isi
MIN_WORDS = 100              # artikel lebih pendek dari ini dibuang
IQR_MULT_UPPER = 3           # batas atas = Q3 + 3*IQR
NEAR_DUP_THRESHOLD = 0.90    # cosine similarity TF-IDF

## 1. Baca data

Tanggal terbit dibaca sebagai UTC. Kalau kosong, dipakai tanggal dari RSS.

In [5]:
df = pd.read_csv(INPUT_FILE)

df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce", utc=True)
rss_date = pd.to_datetime(df["discovery_published"], errors="coerce", utc=True)
df["published_at"] = df["published_at"].fillna(rss_date)

funnel = [("Data awal", len(df))]
print(df.shape)
print(df["status"].value_counts())

(200, 11)
status
success    174
failed      26
Name: count, dtype: int64


## 2. Buang artikel yang gagal

`status` bukan `success`, atau body kosong.

In [6]:
df = df[df["status"] == "success"].copy()
df["body"] = df["body"].fillna("")
df = df[df["body"].str.strip() != ""].copy()

funnel.append(("Setelah buang failed/kosong", len(df)))
print(funnel[-1])

('Setelah buang failed/kosong', 174)


## 3. Filter periode dan zona waktu

Tanggal terbit dikonversi ke WIB karena kurs BI dicatat menurut waktu Indonesia. Artikel tanpa tanggal dibuang (tidak bisa diselaraskan), begitu juga yang di luar 1 Sep 2021 - 1 Sep 2026. Kolom `news_date_wib` dan `published_wib` dipakai nanti untuk penyelarasan dengan hari kerja bursa.

In [7]:
df = df[df["published_at"].notna()].copy()
df["published_wib"] = df["published_at"].dt.tz_convert("Asia/Jakarta")

start = pd.Timestamp(START_DATE, tz="Asia/Jakarta")
end = pd.Timestamp(END_DATE, tz="Asia/Jakarta") + pd.Timedelta(days=1)   # 1 Sep 2026 ikut
df = df[(df["published_wib"] >= start) & (df["published_wib"] < end)].copy()
df["news_date_wib"] = df["published_wib"].dt.strftime("%Y-%m-%d")

funnel.append(("Setelah filter periode", len(df)))
print(funnel[-1], "| rentang:", df["published_wib"].min(), "->", df["published_wib"].max())

('Setelah filter periode', 174) | rentang: 2021-09-02 09:39:44+07:00 -> 2022-06-18 00:13:58+07:00


## 4. Cleaning judul dan isi

Isi artikel masih membawa sisa halaman web. Yang dibersihkan: HTML entity, normalisasi Unicode (NFKC) dan tanda kutip/strip, karakter tak terlihat, URL dan email, boilerplate CNBC ("WATCH:", "Read more", "CNBC Pro", "contributed to this report"), dateline Reuters/AP, paragraf terlalu pendek dan paragraf kembar, serta spasi berlebih.

Boilerplate hanya dicek di paragraf pendek supaya kalimat isi berita yang kebetulan menyebut "CNBC Pro" tidak ikut hilang.

Singkatan bertitik seperti `U.S.`, `U.K.`, `E.U.` diubah jadi `US`, `UK`, `EU`. Ini penting: kalau tidak, tokenizer memecahnya jadi huruf tunggal dan kata "US" di berita geopolitik jadi hilang.

Judul dibersihkan terpisah, termasuk sufiks " - CNBC" yang dibawa oleh RSS.

Di tahap ini teks belum di-lowercase dan tanda baca belum dibuang, karena VADER memakai huruf kapital, tanda seru, dan negasi sebagai sinyal. Versi untuk TF-IDF dibuat di step 9.

In [8]:
QUOTE_MAP = str.maketrans({
    "\u2019": "'", "\u2018": "'",
    "\u201c": '"', "\u201d": '"',
    "\u2013": "-", "\u2014": " - ",
})

INVISIBLE_RE = re.compile(r"[\u200b-\u200f\u202a-\u202e\u2060\ufeff\x00-\x08\x0b\x0c\x0e-\x1f]")
URL_RE       = re.compile(r"https?://\S+|www\.\S+|pic\.twitter\.com/\S+", re.I)
EMAIL_RE     = re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b")
SPACE_RE     = re.compile(r"[ \t\r\f\v]+")
DATELINE_RE  = re.compile(r"^[A-Z][A-Za-z.,' ]{2,40}\s+\((?:Reuters|AP)\)\s*-\s*")
ACRONYM_RE   = re.compile(r"\b(?:[A-Z]\.){2,}")          # U.S., U.K., E.U., U.S.A.
TITLE_SUFFIX_RE = re.compile(r"\s+[-|]\s+CNBC\s*$", re.I)

BOILERPLATE_PATTERNS = [
    r"^(watch|read more|read|related|listen|see also|more from|sign up|subscribe|click here|follow)\b\s*[:\-]",
    r"\bcnbc pro\b",
    r"\bsubscribe to\b",
    r"\bsign up for\b",
    r"\bnewsletter\b",
    r"contributed to this (report|story|article)",
    r"^(disclaimer|copyright)\b",
    r"all rights reserved",
    r"^this (story|article) is (still )?developing",
    r"^(get|receive) the latest",
]
BOILERPLATE_RE = [re.compile(p, re.I) for p in BOILERPLATE_PATTERNS]
BOILERPLATE_MAX_WORDS = 40   # pola di atas hanya berlaku untuk paragraf pendek


def base_normalize(text):
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text)
    text = text.translate(QUOTE_MAP)
    text = INVISIBLE_RE.sub("", text)
    text = ACRONYM_RE.sub(lambda m: m.group().replace(".", ""), text)
    return text


def clean_title(title):
    t = base_normalize(title)
    t = TITLE_SUFFIX_RE.sub("", t)
    return SPACE_RE.sub(" ", t).strip()


def clean_text(text):
    text = base_normalize(text)
    text = URL_RE.sub(" ", text)
    text = EMAIL_RE.sub(" ", text)

    paragraphs = []
    for p in text.split("\n"):
        p = SPACE_RE.sub(" ", p).strip()
        if not p:
            continue
        if not paragraphs:
            p = DATELINE_RE.sub("", p)
        n = len(p.split())
        if n < MIN_PARAGRAPH_WORDS:
            continue
        if n < BOILERPLATE_MAX_WORDS and any(rx.search(p) for rx in BOILERPLATE_RE):
            continue
        paragraphs.append(p)

    paragraphs = list(dict.fromkeys(paragraphs))   # paragraf kembar, urutan tetap
    return "\n".join(paragraphs)


df["title"] = df["title"].fillna(df["discovery_title"]).fillna("")
df["title_clean"] = df["title"].apply(clean_title)
df["body_clean"] = df["body"].apply(clean_text)
df = df[df["body_clean"].str.strip() != ""].copy()

funnel.append(("Setelah cleaning (body tidak kosong)", len(df)))

i = df.index[0]
print("JUDUL   :", df.loc[i, "title"], "->", df.loc[i, "title_clean"])
print("SEBELUM :", df.loc[i, "body"][:300], "\n")
print("SESUDAH :", df.loc[i, "body_clean"][:300])

JUDUL   : Guinea coup rattles iron ore and bauxite markets, stokes economic uncertainty -> Guinea coup rattles iron ore and bauxite markets, stokes economic uncertainty
SEBELUM : A military junta claimed to have seized control in the West African country of Guinea and detained President Alpha Conde, casting uncertainty over key bauxite and iron ore supplies.
The coup, carried out on Sunday by an elite special forces unit led by 41-year-old Col. Mamady Doumbouya, is the lates 

SESUDAH : A military junta claimed to have seized control in the West African country of Guinea and detained President Alpha Conde, casting uncertainty over key bauxite and iron ore supplies.
The coup, carried out on Sunday by an elite special forces unit led by 41-year-old Col. Mamady Doumbouya, is the lates


## 5. Buang artikel template bukan berita geopolitik

CNBC punya banyak artikel rutin yang polanya tetap (daftar saham pilihan, acara Cramer, dst). Isinya soal saham individual, bukan geopolitik. Dibuang lewat pola judul. Daftar ini sengaja pendek dan hanya berisi template yang jelas; sisanya ditangani filter relevansi di step berikutnya.

In [9]:
NOISE_TITLE_PATTERNS = [
    r"stocks? making the biggest moves",
    r"biggest analyst calls",
    r"\bcramer\b",
    r"mad money",
    r"options action",
    r"trading nation",
    r"^watch live",
]
NOISE_TITLE_RE = re.compile("|".join(NOISE_TITLE_PATTERNS), re.I)

is_noise = df["title_clean"].str.contains(NOISE_TITLE_RE)
print("Contoh yang dibuang:")
print(df.loc[is_noise, "title_clean"].head(5).to_string(index=False))

df = df[~is_noise].copy()
funnel.append(("Setelah buang artikel template", len(df)))

Contoh yang dibuang:
Cramer's week ahead: You want to own 'great Ame...
Jim Cramer says owning Devon Energy is an 'insu...
3 things are pulling the market down, but only ...


## 6. Filter relevansi geopolitik

Query di tahap discovery sengaja luas (dolar, suku bunga, dst), jadi banyak artikel yang sebenarnya tidak bicara soal geopolitik. Di sini dicek dengan daftar istilah geopolitik (konflik, sanksi/tarif, aktor dan wilayah). Artikel dipertahankan kalau judulnya memuat minimal satu istilah, atau isinya memuat minimal `GEO_MIN_HITS` istilah *berbeda*. Kata generik seperti "China" atau "oil" sengaja tidak dimasukkan sendirian karena muncul di hampir semua berita bisnis.

Daftar istilah dan ambang ini keputusan tim, jadi cek hasilnya secara manual (contoh artikel yang dibuang dicetak di bawah) dan sesuaikan.

In [10]:
GEO_TERMS = [
    # konflik dan keamanan
    "war", "invasion", "invade", "conflict", "ceasefire", "cease-fire", "military", "troops",
    "missile", "airstrike", "nuclear", "sanctions", "sanction", "embargo", "blockade", "terrorism",
    "escalation", "escalate", "hostilities",
    # diplomasi dan perdagangan strategis
    "geopolitical", "geopolitics", "tariff", "tariffs", "trade war", "export controls",
    "diplomatic", "diplomacy", "summit", "treaty", "nato", "united nations", "opec",
    # aktor dan wilayah
    "ukraine", "russia", "kremlin", "putin", "zelensky", "gaza", "hamas", "israel", "hezbollah",
    "iran", "taiwan", "north korea", "middle east", "red sea", "strait of hormuz",
    "south china sea", "beijing", "xi jinping",
]
GEO_RE = re.compile(
    r"\b(?:" + "|".join(re.escape(t) for t in sorted(GEO_TERMS, key=len, reverse=True)) + r")\b",
    re.I,
)

def geo_hits(text):
    return {m.lower() for m in GEO_RE.findall(text)}

title_hits = df["title_clean"].apply(geo_hits)
all_hits = (df["title_clean"] + " " + df["body_clean"]).apply(geo_hits)
df["geo_hits"] = all_hits.apply(len)

keep = (title_hits.apply(len) >= 1) | (df["geo_hits"] >= GEO_MIN_HITS)

print(f"Lolos: {keep.sum():,} | tidak relevan: {(~keep).sum():,}")
print("\nContoh yang dibuang (cek apakah memang tidak relevan):")
print(df.loc[~keep, "title_clean"].sample(min(10, (~keep).sum()), random_state=0).to_string(index=False))

df = df[keep].copy()
funnel.append(("Setelah filter relevansi geopolitik", len(df)))

Lolos: 137 | tidak relevan: 34

Contoh yang dibuang (cek apakah memang tidak relevan):
World's largest trade deal will come into force...
Why companies are moving to a 'zero trust' mode...
Op-ed: How next-gen investors can build institu...
Global semiconductor sales top half a trillion ...
China applies to join trans-Pacific trade pact ...
India's green hydrogen rush lures companies but...
Crypto scams have cost people more than $1 bill...
London set to miss out on another hot tech list...
Founder of TikTok owner ByteDance steps down as...
                          Elon Musk had a bad week


## 7. Buang outlier panjang artikel

Panjang diukur dalam jumlah kata. Batas bawah tetap (`MIN_WORDS`): di bawah itu biasanya ringkasan satu-dua kalimat atau body yang terambil sebagian. Batas atas memakai IQR (Q3 + 3×IQR) karena distribusi panjang miring ke kanan, dengan sedikit artikel sangat panjang (live blog, transkrip). Dihitung setelah filter relevansi supaya batasnya mengikuti populasi artikel yang benar-benar dipakai.

In [11]:
df["n_words"] = df["body_clean"].str.split().str.len()
print(df["n_words"].describe().round(1))

q1, q3 = df["n_words"].quantile([0.25, 0.75])
lower = MIN_WORDS
upper = q3 + IQR_MULT_UPPER * (q3 - q1)

too_short = (df["n_words"] < lower).sum()
too_long  = (df["n_words"] > upper).sum()
print(f"\nBatas: {lower} - {upper:.0f} kata | terlalu pendek: {too_short}, terlalu panjang: {too_long}")

df = df[(df["n_words"] >= lower) & (df["n_words"] <= upper)].copy()
funnel.append(("Setelah buang outlier panjang", len(df)))

count     137.0
mean      904.2
std       552.7
min       182.0
25%       567.0
50%       750.0
75%      1102.0
max      4808.0
Name: n_words, dtype: float64

Batas: 100 - 2707 kata | terlalu pendek: 0, terlalu panjang: 1


## 8. Buang duplikat

Dicek berlapis: (1) URL sama setelah query string dan `/` akhir dibuang, (2) judul + tanggal sama (judul saja tidak cukup karena CNBC punya judul berulang tiap hari), (3) isi sama persis, (4) isi hampir sama lewat cosine similarity TF-IDF dengan ambang 0.90. Data diurutkan dari yang paling awal terbit, jadi versi pertama yang dipertahankan.

In [12]:
df = df.sort_values("published_at").reset_index(drop=True)

def norm_url(u):
    u = re.sub(r"[?#].*$", "", str(u).lower())
    u = u.replace("://www.", "://")
    return u.rstrip("/")

def norm_text(s):
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

df["_url_key"] = df["url"].apply(norm_url)
df = df.drop_duplicates("_url_key", keep="first")
funnel.append(("Setelah dedup URL", len(df)))

df["_title_key"] = df["title_clean"].apply(norm_text) + "|" + df["news_date_wib"]
df = df.drop_duplicates("_title_key", keep="first")
funnel.append(("Setelah dedup judul+tanggal", len(df)))

df["_body_key"] = df["body_clean"].apply(norm_text)
df = df.drop_duplicates("_body_key", keep="first").reset_index(drop=True)
funnel.append(("Setelah dedup isi persis", len(df)))

In [13]:
# near-duplicate: TF-IDF + cosine similarity, diproses per potongan biar hemat memori
tfidf = TfidfVectorizer(stop_words="english", sublinear_tf=True, max_features=50000)
X = tfidf.fit_transform(df["body_clean"])       # baris sudah L2-normalized -> dot product = cosine

drop = np.zeros(len(df), dtype=bool)
CHUNK = 500

for start_i in range(0, len(df), CHUNK):
    end_i = min(start_i + CHUNK, len(df))
    sim = (X[start_i:end_i] @ X.T).tocsr()
    for r in range(end_i - start_i):
        g = start_i + r
        row = sim.getrow(r)
        similar = row.indices[row.data >= NEAR_DUP_THRESHOLD]
        if (similar < g).any():                 # ada artikel lebih awal yang mirip
            drop[g] = True

print(f"Near-duplicate ditemukan: {drop.sum()}")
df = df[~drop].reset_index(drop=True)
funnel.append(("Setelah dedup near-duplicate", len(df)))

Near-duplicate ditemukan: 0


## 9. Dua versi teks untuk fitur NLP

**Untuk VADER:** pakai `title_clean` dan `body_clean` apa adanya. VADER membaca huruf kapital, tanda seru, dan negasi ("not good"), jadi stopword tidak dibuang.

**Untuk TF-IDF/BoW dan Loughran-McDonald:** `title_bow` dan `body_bow`, yaitu huruf kecil, tanpa angka dan tanda baca, stopword dibuang. Kata negasi (`not`, `no`, `never`, dst) dan `us` (United States) sengaja dipertahankan.

Tidak ada stemming atau lemmatization. Kamus Loughran-McDonald berisi bentuk kata apa adanya (`sanctions`, `sanctioned`, ...), jadi kalau kata diubah ke bentuk dasar, sebagian tidak akan cocok dengan kamus. Kalau nanti mau dipakai untuk TF-IDF saja, bisa ditambahkan terpisah.

In [14]:
NEGATIONS = {"no", "not", "nor", "never", "none", "cannot", "without"}
STOPWORDS = (set(ENGLISH_STOP_WORDS) - NEGATIONS) - {"us"}   # "us" = United States di berita ini

def to_bow_text(text):
    t = text.lower()
    t = t.replace("can't", "cannot").replace("won't", "will not")
    t = re.sub(r"n't\b", " not", t)
    t = re.sub(r"'s\b", "", t)                # possessive
    t = t.replace("'", "")
    tokens = re.findall(r"[a-z]{2,}", t)      # angka dan tanda baca hilang
    return " ".join(w for w in tokens if w not in STOPWORDS)

df["title_bow"] = df["title_clean"].apply(to_bow_text)
df["body_bow"] = df["body_clean"].apply(to_bow_text)

i = 0
print("VADER :", df.loc[i, "body_clean"][:250])
print("BoW   :", df.loc[i, "body_bow"][:250])

VADER : China's health-care sector will probably be the next to fall under scrutiny, analysts warn, as the country's regulators crack down on everything from tech to education to data security .
Chinese President Xi Jinping this week again reiterated the nee
BoW   : china health care sector probably fall scrutiny analysts warn country regulators crack tech education data security chinese president xi jinping week reiterated need support moderate wealth idea common prosperity promoting months driving spate crackd


## 10. Simpan hasil

Tabel di bawah merangkum jumlah artikel yang tersisa di tiap tahap, bisa langsung dipakai di laporan (bagian strategi filtering dan preprocessing).

In [15]:
final_cols = [
    "url", "author", "published_at", "published_wib", "news_date_wib",
    "title_clean", "body_clean",     # untuk VADER
    "title_bow", "body_bow",         # untuk TF-IDF / BoW / Loughran-McDonald
    "geo_hits", "n_words",
]
final_df = df[final_cols].copy()
final_df.to_csv(OUTPUT_FILE, index=False)

summary = pd.DataFrame(funnel, columns=["Tahap", "Jumlah artikel"])
summary["Terbuang"] = -summary["Jumlah artikel"].diff().fillna(0).astype(int)
print(summary.to_string(index=False))

final_df.head()

                               Tahap  Jumlah artikel  Terbuang
                           Data awal             200         0
         Setelah buang failed/kosong             174        26
              Setelah filter periode             174         0
Setelah cleaning (body tidak kosong)             174         0
      Setelah buang artikel template             171         3
 Setelah filter relevansi geopolitik             137        34
       Setelah buang outlier panjang             136         1
                   Setelah dedup URL             136         0
         Setelah dedup judul+tanggal             136         0
            Setelah dedup isi persis             136         0
        Setelah dedup near-duplicate             136         0


,url,author,published_at,published_wib,news_date_wib,title_clean,body_clean,title_bow,body_bow,geo_hits,n_words
0,https://www.cnbc.com/2021/09/02/chinas-regulat...,Weizhen Tan,2021-09-02 02:39:44+00:00,2021-09-02 09:39:44+07:00,2021-09-02,China's health-care sector could be Beijing's ...,China's health-care sector will probably be th...,china health care sector beijing regulatory ta...,china health care sector probably fall scrutin...,2,539
1,https://www.cnbc.com/2021/09/07/guinea-coup-ra...,Elliot Smith,2021-09-07 10:37:21+00:00,2021-09-07 17:37:21+07:00,2021-09-07,Guinea coup rattles iron ore and bauxite marke...,A military junta claimed to have seized contro...,guinea coup rattles iron ore bauxite markets s...,military junta claimed seized control west afr...,3,750
2,https://www.cnbc.com/2021/09/10/us-biden-china...,Evelyn Cheng,2021-09-10 04:36:17+00:00,2021-09-10 11:36:17+07:00,2021-09-10,"Biden, Xi discuss avoiding confrontation in se...",BEIJING - Chinese President Xi Jinping and US ...,biden xi discuss avoiding confrontation second...,beijing chinese president xi jinping us presid...,8,542
3,https://www.cnbc.com/2021/09/15/eu-von-der-ley...,Silvia Amaro,2021-09-15 09:30:36+00:00,2021-09-15 16:30:36+07:00,2021-09-15,EU chief calls for more military independence ...,"LONDON - There is a new international order, w...",eu chief calls military independence amid new ...,london new international order competition fie...,3,783
4,https://www.cnbc.com/2021/09/19/op-ed-will-chi...,Frederick Kempe,2021-09-19 12:00:04+00:00,2021-09-19 19:00:04+07:00,2021-09-19,Op-ed: Will China's President Xi's big bet pay...,Chinese President Xi Jinping is making the mos...,op ed china president xi big bet pay,chinese president xi jinping making audacious ...,7,1050


In [16]:
final_df

,url,author,published_at,published_wib,news_date_wib,title_clean,body_clean,title_bow,body_bow,geo_hits,n_words
0,https://www.cnbc.com/2021/09/02/chinas-regulat...,Weizhen Tan,2021-09-02 02:39:44+00:00,2021-09-02 09:39:44+07:00,2021-09-02,China's health-care sector could be Beijing's ...,China's health-care sector will probably be th...,china health care sector beijing regulatory ta...,china health care sector probably fall scrutin...,2,539
1,https://www.cnbc.com/2021/09/07/guinea-coup-ra...,Elliot Smith,2021-09-07 10:37:21+00:00,2021-09-07 17:37:21+07:00,2021-09-07,Guinea coup rattles iron ore and bauxite marke...,A military junta claimed to have seized contro...,guinea coup rattles iron ore bauxite markets s...,military junta claimed seized control west afr...,3,750
2,https://www.cnbc.com/2021/09/10/us-biden-china...,Evelyn Cheng,2021-09-10 04:36:17+00:00,2021-09-10 11:36:17+07:00,2021-09-10,"Biden, Xi discuss avoiding confrontation in se...",BEIJING - Chinese President Xi Jinping and US ...,biden xi discuss avoiding confrontation second...,beijing chinese president xi jinping us presid...,8,542
3,https://www.cnbc.com/2021/09/15/eu-von-der-ley...,Silvia Amaro,2021-09-15 09:30:36+00:00,2021-09-15 16:30:36+07:00,2021-09-15,EU chief calls for more military independence ...,"LONDON - There is a new international order, w...",eu chief calls military independence amid new ...,london new international order competition fie...,3,783
4,https://www.cnbc.com/2021/09/19/op-ed-will-chi...,Frederick Kempe,2021-09-19 12:00:04+00:00,2021-09-19 19:00:04+07:00,2021-09-19,Op-ed: Will China's President Xi's big bet pay...,Chinese President Xi Jinping is making the mos...,op ed china president xi big bet pay,chinese president xi jinping making audacious ...,7,1050
...,...,...,...,...,...,...,...,...,...,...,...
131,https://www.cnbc.com/2022/05/26/sequoia-coache...,Kate Rooney,2022-05-26 15:14:51+00:00,2022-05-26 22:14:51+07:00,2022-05-26,Sequoia coaches start-ups to cut costs or face...,One of Silicon Valley's most successful ventur...,sequoia coaches start ups cut costs face death...,silicon valley successful venture capital firm...,3,849
132,https://www.cnbc.com/2022/05/27/india-needs-to...,Ravi Buddhavarapu,2022-05-27 00:55:24+00:00,2022-05-27 07:55:24+07:00,2022-05-27,India wants to be the 'pharmacy of the world.'...,India has embarked on an ambitious plan to cut...,india wants pharmacy world wean china,india embarked ambitious plan cut dependence c...,3,938
133,https://www.cnbc.com/select/inflation-when-wil...,Trina Paul,2022-06-08 21:21:11+00:00,2022-06-09 04:21:11+07:00,2022-06-09,"Here's how soon prices could go down again, ac...",Terms apply to American Express benefits and o...,soon prices according experts,terms apply american express benefits offers v...,3,1336
134,https://www.cnbc.com/2022/06/09/quads-maritime...,Ravi Buddhavarapu,2022-06-09 02:55:45+00:00,2022-06-09 09:55:45+07:00,2022-06-09,The Quad's new maritime initiative has potenti...,The Quad countries want to jointly monitor mov...,quad new maritime initiative potential spur mi...,quad countries want jointly monitor movements ...,7,1092
